# APSCHE Term Project: Credit Card Approval Prediction Using IBM Watson Machine Learning
**Course Project for Data Science & Machine Learning Evaluation**

## Executive Summary
This Jupyter Notebook presents an end-to-end Machine Learning solution to automate the credit card approval process for commercial banks. The model predicts whether a credit card application should be **Approved** or **Rejected** based on applicant demographic profiles, financial indicators, employment characteristics, and credit history.

### Technology Stack
- **Core**: Python, Pandas, NumPy
- **Visualizations**: Matplotlib, Seaborn
- **Machine Learning**: Scikit-Learn, XGBoost
- **Serialization**: Joblib
- **Deployment Target**: IBM Watson Machine Learning & Flask Web Application

---

## Epic 1: Data Collection & Data Dictionary
We begin by loading and exploring two datasets:
1. `application_record.csv`: Contains applicant demographics and financial information.
2. `credit_record.csv`: Contains the monthly credit history and payment status of applicants.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Define data directory paths
data_dir = '../dataset'
app_record_path = os.path.join(data_dir, 'application_record.csv')
credit_record_path = os.path.join(data_dir, 'credit_record.csv')

# Load datasets
df_app = pd.read_csv(app_record_path)
df_credit = pd.read_csv(credit_record_path)

print(f"Application Records Shape: {df_app.shape}")
print(f"Credit Records Shape: {df_credit.shape}")

### Dataset Overview & Schema Inspection

In [ ]:
print("=== APPLICATION RECORD COLUMNS ===")
df_app.info()

print("\n=== CREDIT RECORD COLUMNS ===")
df_credit.info()

### Data Head and Tail Inspection

In [ ]:
print("=== APPLICATION RECORD HEAD ===")
display(df_app.head())

print("\n=== CREDIT RECORD HEAD ===")
display(df_credit.head())

### Data Features Explanation

#### Application Record
- `ID`: Unique identifier for each applicant.
- `CODE_GENDER`: Gender of the applicant (`M` = Male, `F` = Female).
- `FLAG_OWN_CAR`: Whether the applicant owns a car (`Y`/`N`).
- `FLAG_OWN_REALTY`: Whether the applicant owns real estate property (`Y`/`N`).
- `CNT_CHILDREN`: Number of children.
- `AMT_INCOME_TOTAL`: Total annual income in local currency.
- `NAME_INCOME_TYPE`: Income source category (e.g., Working, Pensioner, Commercial associate).
- `NAME_EDUCATION_TYPE`: Education level attained (e.g., Secondary, Higher education).
- `NAME_FAMILY_STATUS`: Marital status (e.g., Married, Single, Civil marriage).
- `NAME_HOUSING_TYPE`: Living arrangements (e.g., House/apartment, Rented, With parents).
- `DAYS_BIRTH`: Age in days (negative, counted backwards from today).
- `DAYS_EMPLOYED`: Employment tenure in days (negative; positive value `365243` represents unemployed).
- `FLAG_MOBIL`, `FLAG_WORK_PHONE`, `FLAG_PHONE`, `FLAG_EMAIL`: Binary flags indicating whether mobile, work phone, landline phone, or email was provided.
- `OCCUPATION_TYPE`: Detailed job occupation category.
- `CNT_FAM_MEMBERS`: Family size count.

#### Credit Record
- `ID`: Matches the applicant ID.
- `MONTHS_BALANCE`: The offset month of the record (e.g., 0 is the current month, -1 is one month prior).
- `STATUS`: Credit status for that month:
  - `0`: 1-29 days past due
  - `1`: 30-59 days past due
  - `2`: 60-89 days past due
  - `3`: 90-119 days past due
  - `4`: 120-149 days past due
  - `5`: Overdue or bad debts, write-offs for more than 150 days
  - `C`: Paid off that month
  - `X`: No loan for the month

## Epic 2: Exploratory Data Analysis (EDA)

### 1. Univariate Analysis
We analyze demographics distributions individually to understand our customer base.

In [ ]:
# Gender Distribution Plot
plt.figure(figsize=(6, 4))
sns.countplot(data=df_app, x='CODE_GENDER', palette='pastel')
plt.title('Distribution of Applicants by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

# Income Type Plot
plt.figure(figsize=(10, 5))
sns.countplot(data=df_app, y='NAME_INCOME_TYPE', order=df_app['NAME_INCOME_TYPE'].value_counts().index, palette='Set2')
plt.title('Income Types of Applicants')
plt.xlabel('Count')
plt.ylabel('Income Type')
plt.tight_layout()
plt.show()

# Education Type Plot
plt.figure(figsize=(10, 5))
sns.countplot(data=df_app, y='NAME_EDUCATION_TYPE', order=df_app['NAME_EDUCATION_TYPE'].value_counts().index, palette='Set3')
plt.title('Education Levels of Applicants')
plt.xlabel('Count')
plt.ylabel('Education Level')
plt.tight_layout()
plt.show()

### 2. Family & Housing Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.countplot(ax=axes[0], data=df_app, x='NAME_FAMILY_STATUS', palette='viridis')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title('Family/Marital Status')
axes[0].set_xlabel('')

sns.countplot(ax=axes[1], data=df_app, y='NAME_HOUSING_TYPE', palette='magma')
axes[1].set_title('Housing Type')
axes[1].set_ylabel('')

plt.suptitle('Family Status & Housing Type Distributions', fontsize=16)
plt.tight_layout()
plt.show()

### 3. Income Distribution Analysis
Visualizing applicant incomes to identify skewness and outliers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(ax=axes[0], data=df_app, x='AMT_INCOME_TOTAL', kde=True, bins=30, color='darkblue')
axes[0].set_title('Income Total Distribution')
axes[0].set_xlabel('Income')

sns.boxplot(ax=axes[1], data=df_app, x='AMT_INCOME_TOTAL', color='lightblue')
axes[1].set_title('Boxplot of Income')
axes[1].set_xlabel('Income')

plt.show()
print(df_app['AMT_INCOME_TOTAL'].describe())

## Epic 3: Data Preprocessing & Target Variable Generation

### Business Rules to Determine Credit Approval (Binary Target)
A customer is defined as "Bad" (Target = `0`, Credit Rejected) if they have overdue balance of **60+ days** (Status: `2`, `3`, `4`, or `5`) in any month of their credit record history. Otherwise, they are defined as "Good" (Target = `1`, Credit Approved).

In [ ]:
# Map credit statuses
status_map = {'C': -1, 'X': -1, '0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5}
df_credit['STATUS_NUMERIC'] = df_credit['STATUS'].map(status_map)

# Aggregate: Find maximum risk score reached by each applicant ID
df_max_status = df_credit.groupby('ID')['STATUS_NUMERIC'].max().reset_index()

# Set Target: 0 (Rejected) if status is >= 2, else 1 (Approved)
df_max_status['target'] = df_max_status['STATUS_NUMERIC'].apply(lambda x: 0 if x >= 2 else 1)
print(df_max_status['target'].value_counts())
print(df_max_status['target'].value_counts(normalize=True))

### Merging Application Records and Credit Status Labels

In [ ]:
# Drop duplicate application records
df_app_clean = df_app.drop_duplicates(subset='ID', keep='first')

# Merge
df_merged = pd.merge(df_app_clean, df_max_status[['ID', 'target']], on='ID', how='inner')
print(f"Merged shape: {df_merged.shape}")

### Handling Missing Values & Feature Engineering
1. Transform `DAYS_BIRTH` to `AGE_YEARS`.
2. Transform `DAYS_EMPLOYED` to `EMPLOYMENT_YEARS` and handle `IS_UNEMPLOYED` indicator.
3. Fill `OCCUPATION_TYPE` missing values based on employment status.
4. Drop high-cardinality/redundant features.

In [ ]:
# Missing value inspection
print("Missing values count before handling:")
print(df_merged.isnull().sum())

# Fill occupation
def fill_occupation(row):
    if pd.isna(row['OCCUPATION_TYPE']):
        if row['DAYS_EMPLOYED'] == 365243:
            return 'Pensioner'
        else: 
            return 'Other/Unspecified'
    return row['OCCUPATION_TYPE']

df_merged['OCCUPATION_TYPE'] = df_merged.apply(fill_occupation, axis=1)

# Feature Engineering
df_merged['AGE_YEARS'] = (-df_merged['DAYS_BIRTH'] / 365.25).round(2)
df_merged['IS_UNEMPLOYED'] = (df_merged['DAYS_EMPLOYED'] == 365243).astype(int)
df_merged['EMPLOYMENT_YEARS'] = df_merged['DAYS_EMPLOYED'].apply(lambda x: 0.0 if x == 365243 else (-x / 365.25))
df_merged['EMPLOYMENT_YEARS'] = df_merged['EMPLOYMENT_YEARS'].round(2)

# Drop raw columns
df_engineered = df_merged.drop(columns=['ID', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL'])
print("Missing values count after handling:")
print(df_engineered.isnull().sum())

### Data Splitting and Feature Preprocessing (Encoding & Scaling)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df_engineered.drop(columns=['target'])
y = df_engineered['target']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numerical_cols = ['AMT_INCOME_TOTAL', 'AGE_YEARS', 'EMPLOYMENT_YEARS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS']
categorical_cols = ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 
                      'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']
flag_cols = ['FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'IS_UNEMPLOYED']

# Fit Scaler and Encoder
scaler = StandardScaler()
scaled_train_num = scaler.fit_transform(X_train[numerical_cols])
scaled_test_num = scaler.transform(X_test[numerical_cols])

encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
encoded_train_cat = encoder.fit_transform(X_train[categorical_cols])
encoded_test_cat = encoder.transform(X_test[categorical_cols])
cat_features_names = encoder.get_feature_names(categorical_cols)

# Combine back
X_train_trans = np.hstack((scaled_train_num, encoded_train_cat, X_train[flag_cols].values))
X_test_trans = np.hstack((scaled_test_num, encoded_test_cat, X_test[flag_cols].values))

feature_names = numerical_cols + list(cat_features_names) + flag_cols
print(f"Final features dimension: {X_train_trans.shape}")

### 4. Multivariate Analysis: Feature Correlation Heatmap
Plotting the correlations of numerical features.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_engineered[numerical_cols + ['target']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

## Epic 4: Model Building & Model Comparison
We train 4 classification models:
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. XGBoost Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_curve, auc

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_trans, y_train)
    y_pred = model.predict(X_test_trans)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4)
    })
    
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred))

df_results = pd.DataFrame(results)
display(df_results)

### Model Performance Graph Comparison

In [ ]:
df_melted = df_results.melt(id_vars='Model', var_name='Metric', value_name='Score')
plt.figure(figsize=(10, 6))
sns.barplot(data=df_melted, x='Metric', y='Score', hue='Model', palette='viridis')
plt.ylim(0, 1.05)
plt.title('Model Performance Comparison across Metrics')
plt.show()

### ROC Curve Analysis

In [ ]:
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_prob = model.predict_proba(X_test_trans)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves comparison')
plt.legend(loc='lower right')
plt.show()

### Feature Importance for the Best Model
Plotting feature importances of the XGBoost classifier (the highest F1-Score model).

In [ ]:
xgb_model = models['XGBoost']
importances = xgb_model.feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=np.array(feature_names)[indices], palette='mako')
plt.title('Top 15 Feature Importances (XGBoost)')
plt.xlabel('Relative Importance')
plt.ylabel('Feature Name')
plt.show()

## Save Preprocessing and Best Model Objects
We serialize the fitted scaler, encoder, and best-performing model to deploy them in Flask.

In [ ]:
import joblib
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

joblib.dump(scaler, os.path.join(models_dir, 'scaler.pkl'))
joblib.dump(encoder, os.path.join(models_dir, 'encoder.pkl'))
joblib.dump(models['XGBoost'], os.path.join(models_dir, 'best_model.pkl'))
print("Model, Scaler, and Encoder successfully saved!")

## IBM Watson Machine Learning Deployment Outline

### Step-by-Step Watson Deployment Setup:
1. **IBM Cloud Setup**: Create an account on IBM Cloud, create an **IBM Cloud Object Storage (COS)** instance, and instantiate a **Watson Machine Learning** service.
2. **Watson Studio & Credentials**: Access Watson WML APIs using `ibm-watson-machine-learning` Python SDK and retrieve your API key and instance URL.
3. **Model Storage & Registration**:
   ```python
   from ibm_watson_machine_learning import APIClient
   # Initialize client with credentials
   # Save best_model.pkl into Watson Space
   ```
4. **Scoring Endpoint Creation**: Deploy the registered model online, generating a REST scoring URL to request predictions remotely.

Detailed instructions can be reviewed inside the project's `documentation/ibm_watson_guide.md` file.

### Final Conclusion
This pipeline successfully cleans, pre-processes, and scales the credit approval application records, merges credit history target labels, trains multiple state-of-the-art classifiers, and identifies **XGBoost** as the superior model. It is ready for production scaling via Flask.